In [1]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [2]:
%pip install -q transformers datasets sentencepiece accelerate sacrebleu


import os
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("GPU is not available. Stop before loading mT5.")

# Kaggle working directories
WORK_DIR = "/kaggle/working/ICT4403_MT_Project"
DATA_DIR = os.path.join(WORK_DIR, "data2")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
MODEL_DIR = os.path.join(WORK_DIR, "models")

for folder in [DATA_DIR, RESULTS_DIR, MODEL_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Working directory:", WORK_DIR)


Note: you may need to restart the kernel to use updated packages.
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 15.64
Working directory: /kaggle/working/ICT4403_MT_Project


In [3]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    csv_files = [f for f in files if f.endswith(".csv")]
    if csv_files:
        print("Folder:", root)
        print("CSV files:", csv_files)

Folder: /kaggle/input/datasets/dishita05/gen-ai-ia3-language-translation-dataset
CSV files: ['validation.csv', 'train.csv', 'test.csv']


In [4]:
import os
import pandas as pd

INPUT_DIR = "/kaggle/input/datasets/dishita05/gen-ai-ia3-language-translation-dataset"

train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(INPUT_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}: {len(df)} rows")
    print("Columns:", df.columns.tolist())
    print(df.head(1).to_dict(orient="records"))

assert list(train_df.columns) == ["en", "es"]
assert list(val_df.columns) == ["en", "es"]
assert list(test_df.columns) == ["en", "es"]
assert len(train_df) == 20000
assert len(val_df) == 1000
assert len(test_df) == 1000

print("\nDataset verification passed.")


Train: 20000 rows
Columns: ['en', 'es']
[{'en': "Are you sure it's a kill chip and not a tracker?", 'es': '¿Estás seguro que es un chip asesino y no un rastreador?'}]

Validation: 1000 rows
Columns: ['en', 'es']
[{'en': 'The Pentagon has denied that the activities are part of a direct political advertising campaign, according to published reports.', 'es': 'El Pentágono negó que las actividades sean parte de una campaña de publicidad política directa, de acuerdo con información publicada.'}]

Test: 1000 rows
Columns: ['en', 'es']
[{'en': 'We humans will inevitably end up controlling our own evolution, and, because our power is emergent from nature we will make use of this acquired capacity sooner or later, for better or worse.', 'es': 'Nosotros humanos terminaremos inevitablemente controlando nuestra propia evolución y, puesto que nuestro poder emerge de la naturaleza, tarde o temprano utilizaremos esa capacidad adquirida, para bien o para mal.'}]

Dataset verification passed.


In [5]:
import os
import random
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import Dataset
import sacrebleu

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Model
MODEL_NAME = "google/mt5-small"

# Start with conservative settings for a 15.6 GB T4
MAX_SOURCE_LENGTH = 96
MAX_TARGET_LENGTH = 96
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_EPOCHS = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01

# Save checkpoints and outputs in Kaggle's working directory
MT5_OUTPUT_DIR = os.path.join(MODEL_DIR, "mt5_finetuned")
MT5_RESULTS_DIR = os.path.join(RESULTS_DIR, "mt5")

os.makedirs(MT5_OUTPUT_DIR, exist_ok=True)
os.makedirs(MT5_RESULTS_DIR, exist_ok=True)

print("Model:", MODEL_NAME)
print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))
print("GPU:", torch.cuda.get_device_name(0))
print("Output directory:", MT5_OUTPUT_DIR)

Model: google/mt5-small
Training rows: 20000
Validation rows: 1000
Test rows: 1000
GPU: Tesla T4
Output directory: /kaggle/working/ICT4403_MT_Project/models/mt5_finetuned


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading mT5-small onto GPU...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
)
model = model.to("cuda")

model_params = sum(p.numel() for p in model.parameters())

print("\nModel loaded successfully.")
print("Parameters:", f"{model_params:,}")
print("Device:", next(model.parameters()).device)
print("GPU memory allocated (GB):",
      round(torch.cuda.memory_allocated() / 1e9, 2))

Loading tokenizer...


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Model loaded successfully.
Parameters: 300,176,768
Device: cuda:0
GPU memory allocated (GB): 0.62


In [7]:
from datasets import Dataset

PREFIX = "translate English to Spanish: "

def preprocess_batch(batch):
    inputs = [PREFIX + text for text in batch["en"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True
    )

    targets = tokenizer(
        text_target=batch["es"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = targets["input_ids"]
    return model_inputs

print("Tokenizing training data...")
train_dataset = Dataset.from_pandas(
    train_df[["en", "es"]].reset_index(drop=True)
).map(
    preprocess_batch,
    batched=True,
    remove_columns=["en", "es"],
    desc="Tokenizing training set"
)

print("Tokenizing validation data...")
val_dataset = Dataset.from_pandas(
    val_df[["en", "es"]].reset_index(drop=True)
).map(
    preprocess_batch,
    batched=True,
    remove_columns=["en", "es"],
    desc="Tokenizing validation set"
)

print("\nTokenization complete.")
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Sample tokenized keys:", train_dataset[0].keys())
print("Sample source token count:", len(train_dataset[0]["input_ids"]))
print("Sample target token count:", len(train_dataset[0]["labels"]))

Tokenizing training data...


Tokenizing training set:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing validation data...


Tokenizing validation set:   0%|          | 0/1000 [00:00<?, ? examples/s]


Tokenization complete.
Training examples: 20000
Validation examples: 1000
Sample tokenized keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Sample source token count: 24
Sample target token count: 20


In [8]:
import torch
from transformers import AutoModelForSeq2SeqLM

# Release the currently loaded model
del model
torch.cuda.empty_cache()

# Reload in float32; training will use FP16 mixed precision
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to("cuda")

print("Model ready for training.")
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Parameter dtype:", next(model.parameters()).dtype)
print("Device:", next(model.parameters()).device)
print("GPU memory allocated (GB):",
      round(torch.cuda.memory_allocated() / 1e9, 2))

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model ready for training.
Parameters: 300,176,768
Parameter dtype: torch.float32
Device: cuda:0
GPU memory allocated (GB): 1.2


In [11]:
import inspect
import transformers
from transformers import Seq2SeqTrainingArguments

print("Transformers version:", transformers.__version__)

available_args = inspect.signature(
    Seq2SeqTrainingArguments.__init__
).parameters

for name in [
    "warmup_ratio",
    "warmup_steps",
    "eval_strategy",
    "evaluation_strategy",
    "fp16",
    "gradient_checkpointing",
    "processing_class",
    "tokenizer"
]:
    print(f"{name}: {'supported' if name in available_args else 'not supported'}")

Transformers version: 5.16.1
warmup_ratio: not supported
warmup_steps: supported
eval_strategy: supported
evaluation_strategy: not supported
fp16: supported
gradient_checkpointing: supported
processing_class: not supported
tokenizer: not supported


In [12]:
import inspect
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Build arguments compatible with this installed Transformers version
available_args = inspect.signature(
    Seq2SeqTrainingArguments.__init__
).parameters

args = {
    "output_dir": MT5_OUTPUT_DIR,
    "num_train_epochs": NUM_EPOCHS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "save_total_limit": 1,
    "logging_steps": 100,
    "fp16": True,
    "gradient_checkpointing": True,
    "report_to": "none",
    "seed": SEED,
    "data_seed": SEED
}

if "warmup_ratio" in available_args:
    args["warmup_ratio"] = 0.05

if "eval_strategy" in available_args:
    args["eval_strategy"] = "epoch"
elif "evaluation_strategy" in available_args:
    args["evaluation_strategy"] = "epoch"

if "save_strategy" in available_args:
    args["save_strategy"] = "epoch"

if "predict_with_generate" in available_args:
    args["predict_with_generate"] = False

training_args = Seq2SeqTrainingArguments(**args)

trainer_args = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "data_collator": data_collator
}

trainer_signature = inspect.signature(Seq2SeqTrainer.__init__).parameters

if "processing_class" in trainer_signature:
    trainer_args["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_args["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_args)

print("Trainer configured successfully.")
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Trainer configured successfully.
Training examples: 20000
Validation examples: 1000


In [15]:
import time
import torch

torch.cuda.reset_peak_memory_stats()
training_start = time.time()

train_result = trainer.train()

training_seconds = time.time() - training_start

print("\nTraining complete.")
print(f"Training time: {training_seconds / 60:.2f} minutes")
print(f"Training loss: {train_result.training_loss:.4f}")
print(
    "Peak GPU memory (GB):",
    round(torch.cuda.max_memory_allocated() / 1e9, 2)
)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,39.443354,6.206173


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.
Training time: 26.89 minutes
Training loss: 65.2551
Peak GPU memory (GB): 7.91


In [16]:
import os
import json
import pandas as pd

# Save the final fine-tuned model and tokenizer
trainer.save_model(MT5_OUTPUT_DIR)
tokenizer.save_pretrained(MT5_OUTPUT_DIR)

# Save training metrics
mt5_training_summary = {
    "model": MODEL_NAME,
    "parameters": int(model_params),
    "train_examples": len(train_df),
    "validation_examples": len(val_df),
    "test_examples": len(test_df),
    "epochs": NUM_EPOCHS,
    "training_time_seconds": round(training_seconds, 2),
    "training_time_minutes": round(training_seconds / 60, 2),
    "peak_gpu_memory_gb": round(
        torch.cuda.max_memory_allocated() / 1e9, 2
    ),
    "trainer_reported_training_loss": float(train_result.training_loss),
    "logged_epoch_training_loss": 39.443354,
    "validation_loss": 6.206173,
    "learning_rate": LEARNING_RATE,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "seed": SEED
}

summary_path = os.path.join(
    MT5_RESULTS_DIR, "mt5_training_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(mt5_training_summary, f, indent=2)

print("Model saved to:", MT5_OUTPUT_DIR)
print("Training summary saved to:", summary_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /kaggle/working/ICT4403_MT_Project/models/mt5_finetuned
Training summary saved to: /kaggle/working/ICT4403_MT_Project/results/mt5/mt5_training_summary.json


In [17]:
import os
import time
import pandas as pd
import torch
import sacrebleu
from tqdm.auto import tqdm

# Ensure output directory exists
os.makedirs(MT5_RESULTS_DIR, exist_ok=True)

# Use the fine-tuned model in evaluation mode
model.eval()

test_sources = test_df["en"].fillna("").astype(str).tolist()
test_references = test_df["es"].fillna("").astype(str).tolist()

predictions = []
inference_start = time.time()

BATCH_SIZE = 8

for start in tqdm(
    range(0, len(test_sources), BATCH_SIZE),
    desc="Translating test set"
):
    batch_sources = test_sources[start:start + BATCH_SIZE]
    prefixed_sources = [
        "translate English to Spanish: " + text
        for text in batch_sources
    ]

    inputs = tokenizer(
        prefixed_sources,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    ).to("cuda")

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=1,
            do_sample=False
        )

    batch_predictions = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )
    predictions.extend(batch_predictions)

inference_seconds = time.time() - inference_start

print("\nTranslation complete.")
print("Predictions:", len(predictions))
print(f"Inference time: {inference_seconds:.2f} seconds")
print(f"Average time per sentence: {inference_seconds / len(predictions):.4f} seconds")

Translating test set:   0%|          | 0/125 [00:00<?, ?it/s]


Translation complete.
Predictions: 1000
Inference time: 124.40 seconds
Average time per sentence: 0.1244 seconds


In [19]:
# SacreBLEU expects the references as a list containing one reference corpus
bleu = sacrebleu.corpus_bleu(
    predictions,
    [test_references]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [test_references]
)

print(f"BLEU: {bleu.score:.4f}")
print(f"chrF: {chrf.score:.4f}")

# Save predictions in a separate file
predictions_df = pd.DataFrame({
    "en": test_sources,
    "reference": test_references,
    "prediction": predictions
})

predictions_path = os.path.join(
    MT5_RESULTS_DIR,
    "mt5_predictions.csv"
)
predictions_df.to_csv(predictions_path, index=False)

# Save evaluation metrics
mt5_metrics = {
    "model": MODEL_NAME,
    "test_examples": len(test_sources),
    "BLEU": float(bleu.score),
    "chrF": float(chrf.score),
    "inference_time_seconds": round(inference_seconds, 2),
    "average_inference_time_per_sentence_seconds": round(
        inference_seconds / len(test_sources), 5
    ),
    "parameters": int(model_params)
}

metrics_path = os.path.join(
    MT5_RESULTS_DIR,
    "mt5_metrics.json"
)

import json
with open(metrics_path, "w") as f:
    json.dump(mt5_metrics, f, indent=2)

print("\nPredictions saved:", predictions_path)
print("Metrics saved:", metrics_path)

print("\nSample predictions:")
display(predictions_df.head(5))

BLEU: 1.5833
chrF: 11.4274

Predictions saved: /kaggle/working/ICT4403_MT_Project/results/mt5/mt5_predictions.csv
Metrics saved: /kaggle/working/ICT4403_MT_Project/results/mt5/mt5_metrics.json

Sample predictions:


,en,reference,prediction
0,We humans will inevitably end up controlling o...,Nosotros humanos terminaremos inevitablemente ...,"<extra_id_0> y, porque nuestra capacidad de de..."
1,If she had not rented quarter in this building...,Si no hubiera rentado un cuarto en este edific...,<extra_id_0> a la residencia.
2,I know what the lies were.,Sé cuáles eran las mentiras.,<extra_id_0>.
3,- Here's your buddy for a party...,! - A eso a venido la chavala a la fiesta...,<extra_id_0> -
4,This is not possible for a man of flesh and bl...,Esto no es posible para un hombre de carne y h...,<extra_id_0> a la cabeza.


In [20]:
import torch

model.eval()

diagnostic_sources = test_df["en"].iloc[:5].fillna("").astype(str).tolist()
diagnostic_refs = test_df["es"].iloc[:5].fillna("").astype(str).tolist()

def try_generation(source_text, **generation_kwargs):
    inputs = tokenizer(
        "translate English to Spanish: " + source_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    ).to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            **generation_kwargs
        )

    return tokenizer.decode(output[0], skip_special_tokens=False)

print("Model generation configuration:")
print(model.generation_config)

for i, (src, ref) in enumerate(zip(diagnostic_sources, diagnostic_refs)):
    print(f"\n--- Example {i + 1} ---")
    print("REFERENCE:", ref)
    print("CURRENT:", try_generation(
        src, num_beams=1, do_sample=False
    ))
    print("BEAM SEARCH:", try_generation(
        src, num_beams=4, do_sample=False,
        early_stopping=True
    ))

Model generation configuration:
GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": [
    1
  ],
  "pad_token_id": 0
}


--- Example 1 ---
REFERENCE: Nosotros humanos terminaremos inevitablemente controlando nuestra propia evolución y, puesto que nuestro poder emerge de la naturaleza, tarde o temprano utilizaremos esa capacidad adquirida, para bien o para mal.
CURRENT: <pad> <extra_id_0> y, porque nuestra capacidad de desarrollo y desarrollo, y, porque nuestra capacidad de desarrollo y desarrollo puede ser adquirido en el mundo, y, porque nuestra capacidad de desarrollo, y, porque nuestra capacidad de desarrollo y desarrollo, y, porque nuestra capacidad de desarrollo y desarrollo,
BEAM SEARCH: <pad> <extra_id_0>, y, porque nuestra capacidad de desarrollo y desarrollo, y, porque nuestra capacidad de desarrollo y desarrollo, y, porque nuestra capacidad de desarrollo y desarrollo, y, porque nuestra capacidad de desarrollo y desarrollo de la capacidad de desarrollo, y, porq

In [21]:
# 1. Inspect the original Spanish target text
print("Original target:", train_df["es"].iloc[0])

# 2. Inspect how the target tokenizer encodes it
target_encoding = tokenizer(
    text_target=[train_df["es"].iloc[0]],
    truncation=True,
    max_length=MAX_TARGET_LENGTH
)

print("\nDecoded target labels:")
print(tokenizer.decode(
    target_encoding["input_ids"][0],
    skip_special_tokens=False
))

print("\nTarget labels contain sentinel tokens:",
      any(
          token_id in tokenizer.all_special_ids
          and tokenizer.convert_ids_to_tokens(token_id).startswith("<extra_id_")
          for token_id in target_encoding["input_ids"][0]
      ))

# 3. Inspect the actual prepared training labels
print("\nPrepared training label:")
print(tokenizer.decode(
    train_dataset[0]["labels"],
    skip_special_tokens=False
))

# 4. Check whether the model predicts the sentinel token at the start
first_source = tokenizer(
    "translate English to Spanish: " + train_df["en"].iloc[0],
    return_tensors="pt",
    truncation=True,
    max_length=MAX_SOURCE_LENGTH
).to("cuda")

with torch.inference_mode():
    output = model.generate(
        **first_source,
        max_new_tokens=20,
        num_beams=1,
        do_sample=False
    )

print("\nModel output token IDs:", output[0][:10].tolist())
print("Decoded output:",
      tokenizer.decode(output[0], skip_special_tokens=False))

Original target: ¿Estás seguro que es un chip asesino y no un rastreador?

Decoded target labels:
¿Estás seguro que es un chip asesino y no un rastreador?</s>

Target labels contain sentinel tokens: False

Prepared training label:
¿Estás seguro que es un chip asesino y no un rastreador?</s>

Model output token IDs: [0, 250099, 259, 262, 259, 108186, 291, 1]
Decoded output: <pad> <extra_id_0> a tracker?</s>


In [22]:
import os
import json
import torch

# Inspect training history
print("TRAINING LOGS")
for entry in trainer.state.log_history:
    print(entry)

# Check whether the fine-tuned checkpoint exists
print("\nCHECKPOINT FILES")
for root, dirs, files in os.walk(MT5_OUTPUT_DIR):
    for name in files:
        print(os.path.join(root, name))

# Inspect decoder token probabilities for a sample
sample = tokenizer(
    "translate English to Spanish: Hello, how are you?",
    return_tensors="pt",
    truncation=True,
    max_length=MAX_SOURCE_LENGTH
).to("cuda")

with torch.inference_mode():
    outputs = model(**sample)
    logits = outputs.logits[0, 0]
    probs = torch.softmax(logits.float(), dim=-1)
    top_probs, top_ids = torch.topk(probs, k=10)

print("\nTOP PREDICTED FIRST-TOKEN CANDIDATES")
for token_id, prob in zip(top_ids.tolist(), top_probs.tolist()):
    print(
        repr(tokenizer.convert_ids_to_tokens(token_id)),
        f"{prob:.4f}"
    )

TRAINING LOGS
{'loss': 143.45095703125, 'grad_norm': 3644.38525390625, 'learning_rate': 4.2080000000000004e-05, 'epoch': 0.16, 'step': 100}
{'loss': 76.816650390625, 'grad_norm': 1639.4012451171875, 'learning_rate': 3.408e-05, 'epoch': 0.32, 'step': 200}
{'loss': 53.557236328125, 'grad_norm': 56.82796096801758, 'learning_rate': 2.6079999999999998e-05, 'epoch': 0.48, 'step': 300}
{'loss': 43.9925439453125, 'grad_norm': 45.01799774169922, 'learning_rate': 1.808e-05, 'epoch': 0.64, 'step': 400}
{'loss': 40.7069873046875, 'grad_norm': 42.1267204284668, 'learning_rate': 1.008e-05, 'epoch': 0.8, 'step': 500}
{'loss': 39.4433544921875, 'grad_norm': 36.563331604003906, 'learning_rate': 2.08e-06, 'epoch': 0.96, 'step': 600}
{'eval_loss': 6.206172943115234, 'eval_runtime': 28.5521, 'eval_samples_per_second': 35.024, 'eval_steps_per_second': 4.378, 'epoch': 1.0, 'step': 625}
{'train_runtime': 1613.1374, 'train_samples_per_second': 12.398, 'train_steps_per_second': 0.387, 'total_flos': 11068631224

ValueError: You must specify exactly one of input_ids or inputs_embeds

In [23]:
import torch

model.eval()

sample = tokenizer(
    "translate English to Spanish: Hello, how are you?",
    return_tensors="pt",
    truncation=True,
    max_length=MAX_SOURCE_LENGTH
).to("cuda")

with torch.inference_mode():
    generated = model.generate(
        **sample,
        max_new_tokens=20,
        num_beams=1,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True
    )

print("Generated token IDs:", generated.sequences[0][:10].tolist())
print(
    "Generated tokens:",
    tokenizer.convert_ids_to_tokens(generated.sequences[0][:10].tolist())
)
print(
    "Decoded output:",
    tokenizer.decode(generated.sequences[0], skip_special_tokens=False)
)

# Compare with the saved model's output on the same input
print("\nModel generation config:")
print(model.generation_config)

Generated token IDs: [0, 250099, 261, 4228, 6162, 655, 291, 1]
Generated tokens: ['<pad>', '▁<extra_id_0>', ',', '▁¿', 'qué', '▁es', '?', '</s>']
Decoded output: <pad> <extra_id_0>, ¿qué es?</s>

Model generation config:
GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": [
    1
  ],
  "pad_token_id": 0
}



In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading original pretrained mT5-small...")

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to("cuda")

base_model.eval()

diagnostic_sources = test_df["en"].iloc[:5].fillna("").astype(str).tolist()
diagnostic_refs = test_df["es"].iloc[:5].fillna("").astype(str).tolist()

for i, (src, ref) in enumerate(zip(diagnostic_sources, diagnostic_refs)):
    inputs = tokenizer(
        "translate English to Spanish: " + src,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH
    ).to("cuda")

    with torch.inference_mode():
        output = base_model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=1,
            do_sample=False
        )

    prediction = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print(f"\nExample {i + 1}")
    print("REFERENCE:", ref)
    print("PRETRAINED:", prediction)

del base_model
torch.cuda.empty_cache()

Loading original pretrained mT5-small...


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



Example 1
REFERENCE: Nosotros humanos terminaremos inevitablemente controlando nuestra propia evolución y, puesto que nuestro poder emerge de la naturaleza, tarde o temprano utilizaremos esa capacidad adquirida, para bien o para mal.
PRETRAINED: <extra_id_0> from Spanish to Spanish

Example 2
REFERENCE: Si no hubiera rentado un cuarto en este edificio nuestras vidas habrían sido muy diferentes.
PRETRAINED: <extra_id_0>

Example 3
REFERENCE: Sé cuáles eran las mentiras.
PRETRAINED: <extra_id_0>

Example 4
REFERENCE: ! - A eso a venido la chavala a la fiesta...
PRETRAINED: <extra_id_0>

Example 5
REFERENCE: Esto no es posible para un hombre de carne y hueso.
PRETRAINED: <extra_id_0>


In [25]:
print("Model config:")
print("Model type:", model.config.model_type)
print("Vocabulary size:", model.config.vocab_size)
print("Decoder start token:", model.config.decoder_start_token_id)
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model name/path:", model.config._name_or_path)

print("\nGeneration config:")
print(model.generation_config)

Model config:
Model type: mt5
Vocabulary size: 250112
Decoder start token: 0
Pad token: <pad> 0
EOS token: </s> 1
Tokenizer class: T5Tokenizer
Model name/path: google/mt5-small

Generation config:
GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": [
    1
  ],
  "pad_token_id": 0
}



In [26]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-en-es"

print("Loading English-to-Spanish translation model...")

translation_tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATION_MODEL
)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL
).to("cuda")

translation_model.eval()

sources = test_df["en"].iloc[:5].fillna("").astype(str).tolist()
references = test_df["es"].iloc[:5].fillna("").astype(str).tolist()

for i, (source, reference) in enumerate(zip(sources, references)):
    inputs = translation_tokenizer(
        source,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to("cuda")

    with torch.inference_mode():
        output = translation_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    prediction = translation_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    print(f"\nExample {i + 1}")
    print("SOURCE:    ", source)
    print("REFERENCE: ", reference)
    print("PREDICTION:", prediction)

Loading English-to-Spanish translation model...


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

...


Example 1
SOURCE:     We humans will inevitably end up controlling our own evolution, and, because our power is emergent from nature we will make use of this acquired capacity sooner or later, for better or worse.
REFERENCE:  Nosotros humanos terminaremos inevitablemente controlando nuestra propia evolución y, puesto que nuestro poder emerge de la naturaleza, tarde o temprano utilizaremos esa capacidad adquirida, para bien o para mal.
PREDICTION: Nosotros los humanos inevitablemente terminaremos controlando nuestra propia evolución, y, debido a que nuestro poder emerge de la naturaleza, haremos uso de esta capacidad adquirida tarde o temprano, para bien o para mal.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 2
SOURCE:     If she had not rented quarter in this building our lives would have be very different.
REFERENCE:  Si no hubiera rentado un cuarto en este edificio nuestras vidas habrían sido muy diferentes.
PREDICTION: Si no hubiera alquilado un cuarto en este edificio nuestras vidas habrían sido muy diferentes.

Example 3
SOURCE:     I know what the lies were.
REFERENCE:  Sé cuáles eran las mentiras.
PREDICTION: Sé cuáles eran las mentiras.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 4
SOURCE:     - Here's your buddy for a party...
REFERENCE:  ! - A eso a venido la chavala a la fiesta...
PREDICTION: - Aquí está tu amigo para una fiesta...

Example 5
SOURCE:     This is not possible for a man of flesh and blood.
REFERENCE:  Esto no es posible para un hombre de carne y hueso.
PREDICTION: Esto no es posible para un hombre de carne y hueso.


In [27]:
import os
import json
import time
import torch
import pandas as pd
import sacrebleu
from tqdm.auto import tqdm

# Separate output folder: preserve existing mT5 results
OPUS_RESULTS_DIR = os.path.join(RESULTS_DIR, "opus_mt")
os.makedirs(OPUS_RESULTS_DIR, exist_ok=True)

translation_model.eval()

test_sources = test_df["en"].fillna("").astype(str).tolist()
test_references = test_df["es"].fillna("").astype(str).tolist()

opus_predictions = []
BATCH_SIZE = 16

torch.cuda.synchronize()
start_time = time.time()

for start in tqdm(
    range(0, len(test_sources), BATCH_SIZE),
    desc="Evaluating OPUS-MT"
):
    batch = test_sources[start:start + BATCH_SIZE]

    inputs = translation_tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to("cuda")

    with torch.inference_mode():
        outputs = translation_model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    decoded = translation_tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )
    opus_predictions.extend(decoded)

torch.cuda.synchronize()
opus_inference_seconds = time.time() - start_time

# Corpus-level metrics on the same 1,000 examples
opus_bleu = sacrebleu.corpus_bleu(
    opus_predictions, [test_references]
)
opus_chrf = sacrebleu.corpus_chrf(
    opus_predictions, [test_references]
)

opus_params = sum(
    p.numel() for p in translation_model.parameters()
)

opus_metrics = {
    "model": TRANSLATION_MODEL,
    "model_type": "pretrained_encoder_decoder_translation_baseline",
    "test_examples": len(test_sources),
    "BLEU": float(opus_bleu.score),
    "chrF": float(opus_chrf.score),
    "inference_time_seconds": round(opus_inference_seconds, 2),
    "average_inference_time_per_sentence_seconds": round(
        opus_inference_seconds / len(test_sources), 5
    ),
    "parameters": int(opus_params),
    "beam_size": 4,
    "training_time_seconds": 0
}

predictions_df = pd.DataFrame({
    "en": test_sources,
    "reference": test_references,
    "prediction": opus_predictions
})

predictions_path = os.path.join(
    OPUS_RESULTS_DIR, "opus_mt_predictions.csv"
)
metrics_path = os.path.join(
    OPUS_RESULTS_DIR, "opus_mt_metrics.json"
)

predictions_df.to_csv(predictions_path, index=False)

with open(metrics_path, "w") as f:
    json.dump(opus_metrics, f, indent=2)

print("\nOPUS-MT evaluation complete.")
print(f"BLEU: {opus_bleu.score:.4f}")
print(f"chrF: {opus_chrf.score:.4f}")
print(f"Inference time: {opus_inference_seconds:.2f} seconds")
print(f"Parameters: {opus_params:,}")
print("Predictions saved:", predictions_path)
print("Metrics saved:", metrics_path)

display(predictions_df.head(5))

Evaluating OPUS-MT:   0%|          | 0/63 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


OPUS-MT evaluation complete.
BLEU: 42.5432
chrF: 64.8982
Inference time: 43.27 seconds
Parameters: 77,943,296
Predictions saved: /kaggle/working/ICT4403_MT_Project/results/opus_mt/opus_mt_predictions.csv
Metrics saved: /kaggle/working/ICT4403_MT_Project/results/opus_mt/opus_mt_metrics.json


,en,reference,prediction
0,We humans will inevitably end up controlling o...,Nosotros humanos terminaremos inevitablemente ...,Nosotros los humanos inevitablemente terminare...
1,If she had not rented quarter in this building...,Si no hubiera rentado un cuarto en este edific...,Si no hubiera alquilado un cuarto en este edif...
2,I know what the lies were.,Sé cuáles eran las mentiras.,Sé cuáles eran las mentiras.
3,- Here's your buddy for a party...,! - A eso a venido la chavala a la fiesta...,- Aquí está tu amigo para una fiesta...
4,This is not possible for a man of flesh and bl...,Esto no es posible para un hombre de carne y h...,Esto no es posible para un hombre de carne y h...
